# Your first OpenAI API call

Send a prompt, adjust **temperature**, and read the answer.

Open this notebook in **Jupyter**, **VS Code**, or **Google Colab** with a Python kernel.
Run the cells from top to bottom using **Shift + Enter**. You will need an OpenAI API key
with API billing or credits available. Each run of the request cell makes a paid API call.

## 1. Install the library

Run this once per environment. If you already imported an older version of `openai`, restart
the kernel after installation, then continue below.

In [1]:
%pip install --quiet --upgrade openai

Note: you may need to restart the kernel to use updated packages.


## 2. Connect with your API key

This uses `OPENAI_API_KEY` if you already set it. Otherwise, a hidden input asks for your key.
Paste it into that input, not into the notebook code. The key is kept in this running session.

In [5]:
import os
from getpass import getpass

from openai import OpenAI, OpenAIError
from IPython.display import Markdown, display

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
if not api_key:
    api_key = getpass("Paste your OpenAI API key: ").strip()
if not api_key:
    raise ValueError("No API key entered. Run this cell again and enter your key.")

client = OpenAI(api_key=api_key, timeout=60.0)
print("Client ready. Your key will be checked when you send a request.")

Client ready. Your key will be checked when you send a request.


## 3. Choose your prompt and temperature

Edit the values below:

| Temperature | Try it for |
| --- | --- |
| `0.2` | More focused answers |
| `0.7` | Moderate variation |
| `1.2` | More varied wording and ideas |

The allowed range is **0 to 2**. Temperature changes randomness, not factual accuracy;
even `0` does not guarantee identical answers.

This example uses **GPT-5.4** with reasoning set to `none`, which is required for its
temperature support. Keep that setting when experimenting with temperature.

In [11]:
prompt = """sebutkan rekomendasi model low resource yang mengerti bahasa indonesia untuk keperluan chatbot HRD, dan jelaskan kelebihan dan kekurangannya"""

temperature = 0.7
max_output_tokens = 1000  # Maximum response length in tokens, not words.

## 4. Send your request

Run this cell to call the API and display the answer. To try another temperature, change
the value above, rerun that settings cell, then rerun this one.

In [12]:
if not isinstance(prompt, str) or not prompt.strip():
    raise ValueError("Please enter a non-empty prompt in step 3.")
if isinstance(temperature, bool) or not isinstance(temperature, (int, float)) or not 0 <= temperature <= 2:
    raise ValueError("Temperature must be a number between 0 and 2.")
if isinstance(max_output_tokens, bool) or not isinstance(max_output_tokens, int) or max_output_tokens < 16:
    raise ValueError("max_output_tokens must be a whole number of at least 16.")

response = None
try:
    response = client.responses.create(
        model="gpt-6-luna",
        reasoning={"effort": "none"},
        input=prompt,
        # temperature=temperature,
        max_output_tokens=max_output_tokens,
    )
except OpenAIError as error:
    print(f"The request did not succeed: {error}")
    print("Check your API key, API credits, model access, and internet connection, then try again.")
else:
    display(Markdown(response.output_text or "*No text was returned.*"))
    if response.status != "completed":
        print(f"Response status: {response.status}")
        print(f"Details: {response.incomplete_details}")
        print("If the output token limit was reached, increase max_output_tokens and rerun steps 3 and 4.")

Berikut beberapa model **relatif hemat sumber daya** yang bisa dipertimbangkan untuk chatbot HRD berbahasa Indonesia. “Low resource” di sini terutama berarti dapat dijalankan dengan GPU konsumen atau layanan inferensi ringan—tetap bergantung pada ukuran model, kuantisasi, dan jumlah pengguna serentak.

| Model | Kelebihan | Kekurangan |
|---|---|---|
| **Qwen2.5-3B-Instruct** | Kemampuan multibahasa dan bahasa Indonesia cukup baik untuk ukurannya; instruksi dan format jawaban umumnya rapi; bisa dijalankan secara lokal dengan kuantisasi 4-bit. | Untuk pertanyaan HRD yang rumit, nuansa kebijakan, atau penalaran panjang, akurasinya terbatas; tetap perlu diuji dengan kosakata internal perusahaan. |
| **Qwen2.5-7B-Instruct** | Pilihan seimbang antara kualitas dan kebutuhan komputasi; biasanya lebih baik daripada kelas 3B dalam mengikuti instruksi dan merangkum dokumen; cocok untuk chatbot internal dengan RAG. | Membutuhkan memori dan komputasi lebih besar; jawaban tetap dapat keliru atau mengarang bila dokumen rujukan tidak tersedia atau pencarian gagal. |
| **Llama 3.2 3B Instruct** | Ringan, ekosistem dan dukungan deployment luas; cocok untuk prototipe atau chatbot dengan cakupan pertanyaan yang terbatas. | Kemampuan bahasa Indonesia perlu diuji lebih cermat; dapat kurang konsisten pada istilah HR lokal, bahasa formal, dan pertanyaan kebijakan yang spesifik. |
| **Gemma 2 2B IT** | Sangat kecil dan relatif mudah dijalankan; berguna untuk klasifikasi, routing pertanyaan, atau jawaban singkat yang sederhana. | Kemampuan percakapan dan bahasa Indonesianya lebih terbatas dibanding model yang lebih besar; kurang ideal sebagai satu-satunya model untuk konsultasi HRD. |
| **SeaLLM 7B (Instruct, jika tersedia dan lisensinya sesuai)** | Dikembangkan dengan fokus pada bahasa Asia Tenggara, sehingga layak diuji untuk bahasa Indonesia dan konteks regional. | Pilihan versi, dukungan, dan ekosistem deployment bisa lebih terbatas daripada Qwen atau Llama; cek lisensi dan hasil benchmark versi yang akan digunakan. |
| **Model bahasa Indonesia atau Melayu yang di-fine-tune komunitas** | Bisa lebih cocok untuk gaya bahasa lokal dan istilah Indonesia tertentu; berpotensi baik untuk tugas yang sempit. | Kualitas, dokumentasi, pembaruan, keamanan, dan lisensinya sangat bervariasi. Jangan memilih hanya berdasarkan klaim “mendukung bahasa Indonesia”; uji langsung pada data perusahaan. |

### Rekomendasi praktis

- **Untuk mulai dengan biaya rendah:** uji **Qwen2.5-3B-Instruct** dengan kuantisasi 4-bit.
- **Untuk keseimbangan kualitas dan biaya:** uji **Qwen2.5-7B-Instruct** dengan kuantisasi 4-bit.
- **Untuk tugas sangat sederhana** seperti mengarahkan pertanyaan ke kategori cuti, payroll, atau benefit: model 2–3B dapat cukup, atau gunakan model kecil hanya untuk routing.
- Jika memakai API, bandingkan juga model kecil yang tersedia di penyedia pilihan Anda. Biaya dan latensi aktual bisa lebih penting daripada ukuran model.

### Hal penting untuk chatbot HRD

Sebaiknya chatbot **tidak mengandalkan ingatan model** untuk menjawab kebijakan perusahaan. Gunakan **RAG**: cari jawaban dari dokumen HRD resmi—misalnya peraturan cuti, benefit, payroll, dan prosedur—lalu minta model menjawab berdasarkan dokumen tersebut serta menyebutkan sumbernya. Tambahkan pula:

- pembatasan akses untuk dokumen yang sensitif;
- aturan agar bot mengakui jika jawaban tidak ditemukan dan mengarahkan ke staf HR;
- evaluasi dengan pertanyaan nyata dalam bahasa Indonesia, termasuk variasi informal dan istilah internal;
- pemeriksaan manusia untuk topik berisiko tinggi seperti PHK, disiplin, kompensasi, atau keluhan karyawan.

**Pilihan awal yang paling masuk akal:** bandingkan Qwen2.5-3B dan Qwen2.5-7B pada kumpulan pertanyaan HRD Anda sendiri. Kinerja bahasa Indonesia dan kepatuhan terhadap kebijakan perusahaan sebaiknya diukur langsung, bukan diasumsikan dari ukuran atau nama model.

## Try it yourself

- Ask for a Python explanation, a product name, or a short story.
- Keep the same prompt and compare `0.2` with `1.2` by rerunning steps 3 and 4.
- Run several times to observe variation; one pair of answers is only an illustration.

**Sharing this notebook:** saved cell outputs can contain your prompts or generated text.
Clear outputs before sharing if they contain private information.

Reference: [Official OpenAI documentation — GPT-5.4 parameter compatibility](https://developers.openai.com/api/docs/guides/latest-model?model=gpt-5.4).